In [ ]:
!pip install -q pandas
!pip install -q html2text
!pip install -q pyarrow

In [ ]:
filename = 'spam-labels-merged.parquet.gzip'

In [ ]:
import gc
import re
import pandas as pd
import html2text

In [ ]:
# helpers
def split_group_lines(text, tokens = 2000):
    lines = text.splitlines(keepends=True)
    if len(lines) == 1:
        return lines
    outputs = []
    tmp = ""
    for line in lines:
        tmp = tmp + line
        if len(tmp) > tokens:
            outputs.append(tmp)
            tmp = ""
    else:
        if tmp and len(outputs) == 0:
            return [tmp]
        outputs[-1] = outputs[-1] + tmp
            
    return outputs


h = html2text.HTML2Text()
h.ignore_links = True
h.images_to_alt = True

def strip_html(html):
    return h.handle(html)

In [ ]:
data = pd.read_parquet(filename)

In [ ]:
%%time

data['text'] = data.apply(
            lambda row: 
                split_group_lines(
                str(row['title']) + '\n\n' + str(row['summary']) + '\n\n' + strip_html(str(row['content']))
                if row['summary_customized']==1 else (str(row['title']) + '\n\n'  + strip_html(str(row['content']))))
            ,
            axis=1
        )

In [ ]:
exploded = data.explode('text')

In [ ]:
result = exploded[['article_id', 'text', 'is_spam']]

In [ ]:
result

In [ ]:
result.to_parquet('processed-' + filename, compression='gzip') 